# 9. Inventario de secretos KV v2

Este notebook carga un conjunto de secretos en un Vault Enterprise y deja un script que los inventaría por la API.

El lote son **100 secretos KV v2** repartidos en **15 namespaces** anidados. Cada namespace tiene más de un mount KV v2. **75 secretos** llevan metadatos de atribución (`owner`, `app`, `project`, `platform`, `env`, `team`). Los otros **25** se crean sin `custom_metadata`.

El script `scripts/vault_secret_inventory.sh` recorre namespaces, mounts y paths, y escribe un CSV. Marca los secretos sin metadatos y los que son más viejos que `SECRET_MAX_AGE_DAYS`. Para Secrets Sync anota si el secreto está asociado y dónde: cuenta y región de AWS, Key Vault de Azure, proyecto y regiones de GCP, repositorio de GitHub o proyecto de Vercel. Este notebook no crea destinos. La sesión es la del CLI de Vault.


In [ ]:
! pip install pandas Jinja2 plotly ipywidgets anywidget

In [ ]:
%env VAULT_ADDR=
%env VAULT_TOKEN=
%env VAULT_SKIP_VERIFY=true

env: VAULT_ADDR=https://vault-vm.jose-merchan.sbx.hashidemos.io
env: VAULT_TOKEN=<tu-token>
env: VAULT_SKIP_VERIFY=true


## Lote de secretos

El árbol sigue el esquema `/ns1`, `/ns1/ns11`, `/ns1/ns12/ns121`, `/ns2`, y así hasta 15 namespaces. El padre se crea antes que el hijo.

| Namespace | Mounts KV v2 | Secretos |
| --- | --- | --- |
| `/ns1` | apps, shared, platform | 10 |
| `/ns1/ns11` | apps, config | 6 |
| `/ns1/ns12` | apps, shared | 7 |
| `/ns1/ns12/ns121` | runtime, config | 5 |
| `/ns2` | apps, data | 8 |
| `/ns2/ns21` | apps, config | 6 |
| `/ns2/ns22` | shared, platform | 6 |
| `/ns3` | secrets, config | 8 |
| `/ns3/ns31` | apps, config | 6 |
| `/ns3/ns31/ns311` | pci, runtime | 5 |
| `/ns4` | apps, shared | 8 |
| `/ns4/ns41` | apps, config | 7 |
| `/ns5` | apps, platform | 8 |
| `/ns5/ns51` | shared, config | 5 |
| `/ns5/ns51/ns511` | runtime, apps | 5 |

Dentro de cada mount los paths alternan cuatro formas: `postgres/credentials`, `api/token`, `teams/<servicio>/webhook` y `config/feature-flags`.

El valor de cada secreto es `demo-N` y no se imprime. Los secretos cuyo ordinal es múltiplo de 4 se quedan sin metadatos custom (25). Los otros 75 reciben `owner`, `app`, `project`, `platform`, `env` y `team`.

Volver a ejecutar la celda reutiliza namespaces y mounts. `vault kv put` añade una versión, así que la edad del secreto vuelve a contar desde esa escritura.


In [2]:
%%bash
#set -euo pipefail
#set -a
#source /tmp/vault/config.env
#set +a
#unset VAULT_NAMESPACE

ns_exists() {
  local full=$1
  local parent base json
  parent=$(dirname "$full")
  base=$(basename "$full")
  if [[ "$parent" == "." ]]; then
    json=$(vault namespace list -format=json 2>/dev/null || true)
  else
    json=$(VAULT_NAMESPACE="$parent" vault namespace list -format=json 2>/dev/null || true)
  fi
  [[ -n "$json" ]] || return 1
  printf '%s' "$json" | jq -e --arg k "${base}/" '
    if type == "array" then index($k) != null
    elif (.keys? != null) then (.keys | index($k) != null)
    elif (.data.keys? != null) then (.data.keys | index($k) != null)
    else false end
  ' >/dev/null
}

create_namespace() {
  local full=$1
  local parent base
  parent=$(dirname "$full")
  base=$(basename "$full")
  if ns_exists "$full"; then
    echo "namespace /${full} ya existe"
    return 0
  fi
  if [[ "$parent" == "." ]]; then
    vault namespace create "$base" >/dev/null
  else
    VAULT_NAMESPACE="$parent" vault namespace create "$base" >/dev/null
  fi
  echo "namespace /${full} creado"
}

enable_kv() {
  local ns=$1
  local mount=$2
  local listed typ ver
  listed=$(VAULT_NAMESPACE="$ns" vault secrets list -format=json 2>/dev/null || echo '{}')
  if printf '%s' "$listed" | jq -e --arg m "${mount}/" 'has($m)' >/dev/null; then
    typ=$(printf '%s' "$listed" | jq -r --arg m "${mount}/" '.[$m].type // ""')
    ver=$(printf '%s' "$listed" | jq -r --arg m "${mount}/" '((.[$m].options // {}).version // "1") | tostring')
    if [[ "$typ" != "kv" || "$ver" != "2" ]]; then
      echo "El mount /${ns}/${mount} existe y no es KV v2 (type=${typ} version=${ver})." >&2
      exit 1
    fi
    echo "mount /${ns}/${mount} ya existe"
    return 0
  fi
  VAULT_NAMESPACE="$ns" vault secrets enable -path="$mount" kv-v2 >/dev/null
  echo "mount /${ns}/${mount} creado"
}

secret_path_for() {
  local idx=$1
  local service=$2
  case "$idx" in
    0) printf 'postgres/credentials' ;;
    1) printf 'api/token' ;;
    2) printf 'teams/%s/webhook' "$service" ;;
    3) printf 'config/feature-flags' ;;
    *) printf 'extra/%s/%s' "$service" "$idx" ;;
  esac
}

plans=(
  "ns1|apps shared platform|10"
  "ns1/ns11|apps config|6"
  "ns1/ns12|apps shared|7"
  "ns1/ns12/ns121|runtime config|5"
  "ns2|apps data|8"
  "ns2/ns21|apps config|6"
  "ns2/ns22|shared platform|6"
  "ns3|secrets config|8"
  "ns3/ns31|apps config|6"
  "ns3/ns31/ns311|pci runtime|5"
  "ns4|apps shared|8"
  "ns4/ns41|apps config|7"
  "ns5|apps platform|8"
  "ns5/ns51|shared config|5"
  "ns5/ns51/ns511|runtime apps|5"
)

owners=(alice.nguyen bob.silva carol.okonkwo diego.martinez elena.rossi farid.hassan grace.chen hugo.berg)
apps_md=(checkout ledger identity radar billing notifications onboarding fraud)
projects=(transform atlas northstar lighthouse horizon)
platforms=(kubernetes nomad lambda vm)
envs=(dev staging prod sandbox)
teams=(payments platform data security retail)
services=(checkout ledger identity radar billing notify onboard fraud stores analytics)

ns_total=0
mount_total=0
secret_total=0
meta_total=0
plain_total=0
n=0
seen=$(mktemp)
trap 'rm -f "$seen"' EXIT

for plan in "${plans[@]}"; do
  IFS='|' read -r ns mounts_s count <<< "$plan"
  create_namespace "$ns"
  ns_total=$((ns_total + 1))
  mounts=($mounts_s)
  if [[ "$ns" == "ns2" ]]; then
    listed=$(VAULT_NAMESPACE="$ns" vault secrets list -format=json 2>/dev/null || echo '{}')
    if printf '%s' "$listed" | jq -e 'has("lake/")' >/dev/null; then
      VAULT_NAMESPACE="$ns" vault secrets disable lake >/dev/null
      echo "mount /${ns}/lake eliminado"
    fi
  fi
  nmounts=${#mounts[@]}
  mi=0
  while [[ $mi -lt $nmounts ]]; do
    enable_kv "$ns" "${mounts[$mi]}"
    mount_total=$((mount_total + 1))
    mi=$((mi + 1))
  done

  i=0
  while [[ $i -lt $count ]]; do
    n=$((n + 1))
    mount=${mounts[$((i % nmounts))]}
    local_idx=$((i / nmounts))
    service=${services[$((n % ${#services[@]}))]}
    path=$(secret_path_for "$local_idx" "$service")
    if [[ "$local_idx" == "0" ]]; then
      VAULT_NAMESPACE="$ns" vault kv metadata delete -mount="$mount" database/credentials >/dev/null 2>&1 || true
    fi
    key="${ns}|${mount}|${path}"
    if grep -Fxq "$key" "$seen"; then
      echo "Path duplicado: ${key}" >&2
      exit 1
    fi
    printf '%s\n' "$key" >> "$seen"

    VAULT_NAMESPACE="$ns" vault kv put -mount="$mount" "$path" value="demo-${n}" >/dev/null
    secret_total=$((secret_total + 1))

    if (( n % 4 == 0 )); then
      plain_total=$((plain_total + 1))
      echo "secreto ${n} /${ns}/${mount}/${path} sin metadatos"
    else
      owner=${owners[$((n % ${#owners[@]}))]}
      app=${apps_md[$((n % ${#apps_md[@]}))]}
      project=${projects[$((n % ${#projects[@]}))]}
      platform=${platforms[$((n % ${#platforms[@]}))]}
      env_md=${envs[$((n % ${#envs[@]}))]}
      team=${teams[$((n % ${#teams[@]}))]}
      VAULT_NAMESPACE="$ns" vault kv metadata put -mount="$mount" \
        -custom-metadata="owner=${owner}" \
        -custom-metadata="app=${app}" \
        -custom-metadata="project=${project}" \
        -custom-metadata="platform=${platform}" \
        -custom-metadata="env=${env_md}" \
        -custom-metadata="team=${team}" \
        "$path" >/dev/null
      meta_total=$((meta_total + 1))
      echo "secreto ${n} /${ns}/${mount}/${path} owner=${owner} app=${app} project=${project} platform=${platform} env=${env_md} team=${team}"
    fi
    i=$((i + 1))
  done
done

echo
echo "namespaces=${ns_total}"
echo "kv_mounts=${mount_total}"
echo "secretos=${secret_total}"
echo "con_metadatos=${meta_total}"
echo "sin_metadatos=${plain_total}"

[[ "$ns_total" -eq 15 ]]
[[ "$mount_total" -eq 31 ]]
[[ "$secret_total" -eq 100 ]]
[[ "$meta_total" -eq 75 ]]
[[ "$plain_total" -eq 25 ]]


namespace /ns1 ya existe
mount /ns1/apps ya existe
mount /ns1/shared ya existe
mount /ns1/platform ya existe
secreto 1 /ns1/apps/database/credentials owner=bob.silva app=ledger project=atlas platform=nomad env=staging team=platform
secreto 2 /ns1/shared/database/credentials owner=carol.okonkwo app=identity project=northstar platform=lambda env=prod team=data
secreto 3 /ns1/platform/database/credentials owner=diego.martinez app=radar project=lighthouse platform=vm env=sandbox team=security
secreto 4 /ns1/apps/api/token sin metadatos
secreto 5 /ns1/shared/api/token owner=farid.hassan app=notifications project=transform platform=nomad env=staging team=payments
secreto 6 /ns1/platform/api/token owner=grace.chen app=onboarding project=atlas platform=lambda env=prod team=platform
secreto 7 /ns1/apps/teams/fraud/webhook owner=hugo.berg app=fraud project=northstar platform=vm env=sandbox team=data
secreto 8 /ns1/shared/teams/stores/webhook sin metadatos
secreto 9 /ns1/platform/teams/analytics/

El árbol se vuelve a leer de la API. La primera línea es el namespace raíz `/`. Cada línea lista los mounts KV v2 de ese namespace.


In [ ]:
%%bash
#set -euo pipefail
#set -a
#source /tmp/vault/config.env
#set +a
#unset VAULT_NAMESPACE

list_children() {
  local parent=$1
  local json
  if [[ -z "$parent" ]]; then
    json=$(vault namespace list -format=json 2>/dev/null || true)
  else
    json=$(VAULT_NAMESPACE="$parent" vault namespace list -format=json 2>/dev/null || true)
  fi
  [[ -n "$json" ]] || return 0
  printf '%s' "$json" | jq -r '
    if type == "array" then .[]
    elif (.keys? != null) then .keys[]
    elif (.data.keys? != null) then .data.keys[]
    else empty end
  ' | sed 's:/$::' | sort
}

kv_mounts() {
  local ns=$1
  if [[ -z "$ns" ]]; then
    vault secrets list -format=json
  else
    VAULT_NAMESPACE="$ns" vault secrets list -format=json
  fi | jq -r '
    to_entries[]
    | select(.value.type == "kv")
    | select((((.value.options // {}).version) // "1" | tostring) == "2")
    | .key
  ' | tr '\n' ' '
}

print_tree() {
  local parent=$1
  local depth=$2
  local child full kv
  while IFS= read -r child; do
    [[ -n "$child" ]] || continue
    if [[ -z "$parent" ]]; then
      full=$child
    else
      full="${parent}/${child}"
    fi
    case "$full" in
      ns1|ns1/*|ns2|ns2/*|ns3|ns3/*|ns4|ns4/*|ns5|ns5/*) ;;
      *) continue ;;
    esac
    kv=$(kv_mounts "$full")
    printf '%*s/%s   kv-v2: %s\n' $((depth * 2)) "" "$full" "$kv"
    print_tree "$full" $((depth + 1))
  done < <(list_children "$parent")
}

printf '/   kv-v2: %s\n' "$(kv_mounts "")"
print_tree "" 1


/   kv-v2: <mounts del namespace raíz omitidos> 
  /ns1   kv-v2: apps/ platform/ shared/ 
    /ns1/ns11   kv-v2: apps/ config/ 


## Script de inventario

La celda siguiente escribe [scripts/vault_secret_inventory.sh](scripts/vault_secret_inventory.sh). Las llamadas las hace el CLI de Vault (`vault list` y `vault read`). Dirección, token y CA salen de esa configuración: `VAULT_ADDR`, `VAULT_TOKEN`, `VAULT_CACERT`, `VAULT_SKIP_VERIFY`, el fichero de token o el agent. El script no abre otro login.

Recorrido:

1. `vault list sys/namespaces`, de la raíz hacia abajo.
2. `vault read sys/mounts` y se queda con los mounts `kv` de versión 2.

De cada mount se copia el campo `local` a `mount_local`. `true` es un KV local al cluster: performance replication no replica sus datos. `false` es un mount replicable.
3. `vault list <mount>/metadata` y, si la clave termina en `/`, baja a esa carpeta.
4. `vault read <mount>/metadata/<path>` para tiempos y `custom_metadata`.
5. `vault list sys/sync/destinations`, `vault read` de cada destino y `vault read .../associations`. Un secreto está sincronizado cuando su accessor y su path aparecen en alguna asociación. Del destino se lee el sitio, sin las credenciales (Vault las enmascara).

El CSV sale con estas columnas:

`namespace`, `secret_mount`, `mount_local`, `secret_path`, `absolute_path`, `created_time`, `updated_time`, `age_seconds`, `age_days`, `max_age_days`, `older_than_threshold`, `missing_metadata`, `owner`, `app`, `project`, `platform`, `env`, `team`, `custom_metadata`, `secret_sync`, `sync_destinations`, `sync_where`, `sync_targets`, `sync_api`.

`absolute_path` junta namespace, mount y path: `/ns1/ns12/apps/postgres/credentials`.

`missing_metadata=true` cuando `custom_metadata` es null o `{}`.

`updated_time` es la fecha de la versión actual. `older_than_threshold=true` cuando la más reciente entre `created_time` y esa versión es anterior a ahora menos `SECRET_MAX_AGE_DAYS` días. Un cambio solo de metadatos custom no rejuvenece el secreto. La comparación usa segundos: con umbral `0`, cualquier secreto ya escrito queda marcado.

`secret_sync=true` si hay al menos una asociación. `sync_destinations` es `tipo/nombre:estado`. `sync_where` describe el sitio y `sync_targets` es el mismo dato en JSON. `false` si la API respondió y no hay asociaciones. `unknown` si el endpoint no existe, la feature no está activa o el token no puede leerlo; entonces `sync_api=unavailable`.

Qué entra en `sync_where`, según el tipo de destino:

| Destino | Sitio |
| --- | --- |
| AWS Secrets Manager (`aws-sm`) | `account` (sale del `role_arn`; las access keys vienen enmascaradas y no traen la cuenta), `region` |
| Azure Key Vault (`azure-kv`) | `key_vault`, `key_vault_uri`, `tenant`, `cloud`. `region` solo si el destino la guarda; la API de sync identifica el vault por su URI |
| GCP Secret Manager (`gcp-sm`) | `project`, `region` (las `replication_locations`) |
| GitHub (`gh`) | organización o owner, repositorio, environment, URL enterprise |
| Vercel (`vercel-project`) | `project_id`, `team_id`, entornos de deployment |

```bash
SECRET_MAX_AGE_DAYS=90 ./scripts/vault_secret_inventory.sh --output /tmp/vault/secret-inventory.csv
./scripts/vault_secret_inventory.sh --max-age-days 30 --output inventario.csv
```

El CSV incluye todos los KV v2 que el token puede leer, también los de otros notebooks. La vista muestra el namespace raíz `/` y el árbol `/ns1` … `/ns5`. Los conteos del lote (100 secretos, 15 namespaces, 25 sin metadatos) se comprueban solo sobre `/ns1` … `/ns5`.


In [ ]:
%%bash
set -euo pipefail
mkdir -p scripts
cat > scripts/vault_secret_inventory.sh << 'INVENTORY_SCRIPT'
#!/usr/bin/env bash
# Inventario de secretos KV v2: namespace, mount, path, metadatos y Secrets Sync.
#
# Uso:
#   SECRET_MAX_AGE_DAYS=90 ./scripts/vault_secret_inventory.sh
#   SECRET_MAX_AGE_DAYS=90 ./scripts/vault_secret_inventory.sh --output /tmp/vault/secret-inventory.csv
#   ./scripts/vault_secret_inventory.sh --max-age-days 30 --output inventario.csv
#
# La sesión de Vault es la del CLI: VAULT_ADDR, VAULT_TOKEN, VAULT_CACERT,
# VAULT_SKIP_VERIFY, el fichero de token o el agent. El script no fija otro login.
#
#   SECRET_MAX_AGE_DAYS o --max-age-days
#     Edad en días. older_than_threshold=true cuando la actividad más reciente
#     del secreto (creación o última versión) es anterior a ahora menos esos días.
#
# El script no configura Secrets Sync. Lee los destinos y, para cada secreto
# asociado, anota dónde vive: cuenta y región de AWS, Key Vault de Azure,
# proyecto y regiones de GCP Secret Manager, repositorio de GitHub o proyecto
# de Vercel. Si el endpoint no existe o la feature no está activa,
# secret_sync queda en "unknown".

set -euo pipefail

usage() {
  cat <<'EOF'
Uso:
  SECRET_MAX_AGE_DAYS=<dias> vault_secret_inventory.sh [--output fichero.csv]
  vault_secret_inventory.sh --max-age-days <dias> [--output fichero.csv]

Usa el CLI de Vault ya configurado (dirección, token y CA) para recorrer los
namespaces visibles, listar los mounts KV v2 y escribir un CSV con:

  namespace, secret_mount, mount_local, secret_path, absolute_path,
  created_time, updated_time, age_seconds, age_days, max_age_days,
  older_than_threshold, missing_metadata,
  owner, app, project, platform, env, team, custom_metadata,
  secret_sync, sync_destinations, sync_where, sync_targets, sync_api

older_than_threshold es true cuando el secreto es más viejo que
SECRET_MAX_AGE_DAYS. La edad sale de la marca más reciente entre la creación
del secreto y la versión actual.

missing_metadata es true cuando custom_metadata es null o está vacío.

mount_local es el campo local del mount en sys/mounts. true significa que el
KV es local al cluster: performance replication no replica sus datos. false
es un mount replicable.

secret_sync es true si el secreto tiene alguna asociación. sync_where dice
el sitio externo de cada una. sync_targets es el mismo dato en JSON.
Si Secrets Sync no responde, secret_sync queda en unknown.

En AWS, account sale del role_arn del destino (las access keys se devuelven
enmascaradas y no traen la cuenta). region es el campo region del destino.
En Azure, key_vault y key_vault_uri identifican el vault; region solo aparece
si el destino la guarda. En GCP, project es project_id y locations son las
replication_locations.
EOF
}

MAX_AGE_DAYS=""
OUTPUT=""
TMP_DIR=""
NS_FILE=""
SYNC_INDEX=""
SYNC_API_FILE=""
CSV_TMP=""
VAULT_API_HTTP=""
VAULT_API_BODY=""
TOTAL=0
MISSING=0
STALE=0
SYNCED=0
SYNC_UNKNOWN=0

parse_args() {
  MAX_AGE_DAYS="${SECRET_MAX_AGE_DAYS:-}"
  OUTPUT=""
  while [[ $# -gt 0 ]]; do
    case "$1" in
      --max-age-days)
        [[ $# -ge 2 ]] || { echo "Falta el valor de --max-age-days" >&2; exit 2; }
        MAX_AGE_DAYS=$2
        shift 2
        ;;
      --output)
        [[ $# -ge 2 ]] || { echo "Falta el valor de --output" >&2; exit 2; }
        OUTPUT=$2
        shift 2
        ;;
      -h|--help)
        usage
        exit 0
        ;;
      *)
        echo "Argumento no reconocido: $1" >&2
        usage >&2
        exit 2
        ;;
    esac
  done

  if [[ -z "$MAX_AGE_DAYS" ]]; then
    echo "Indica la edad en días con SECRET_MAX_AGE_DAYS o --max-age-days." >&2
    usage >&2
    exit 2
  fi
  if ! [[ "$MAX_AGE_DAYS" =~ ^[0-9]+$ ]]; then
    echo "La edad tiene que ser un entero >= 0. Recibido: ${MAX_AGE_DAYS}" >&2
    exit 2
  fi
  if [[ -z "$OUTPUT" ]]; then
    OUTPUT="${INVENTORY_CSV:-secret-inventory.csv}"
  fi

  local bin
  for bin in vault jq awk; do
    command -v "$bin" >/dev/null 2>&1 || { echo "Falta el comando ${bin}" >&2; exit 1; }
  done
}

prepare_workspace() {
  TMP_DIR=$(mktemp -d)
  NS_FILE="${TMP_DIR}/namespaces"
  SYNC_INDEX="${TMP_DIR}/sync.jsonl"
  SYNC_API_FILE="${TMP_DIR}/sync-api.tsv"
  CSV_TMP="${TMP_DIR}/inventory.csv"
  trap 'rm -rf "$TMP_DIR"' EXIT
  touch "$SYNC_INDEX" "$SYNC_API_FILE"
  TOTAL=0
  MISSING=0
  STALE=0
  SYNCED=0
  SYNC_UNKNOWN=0
}

display_ns() {
  if [[ -z "${1:-}" ]]; then
    printf '/'
  else
    printf '/%s' "$1"
  fi
}

join_path() {
  local result="" part
  for part in "$@"; do
    [[ -n "$part" ]] || continue
    if [[ -z "$result" ]]; then
      result=$part
    else
      result="${result}/${part}"
    fi
  done
  printf '%s' "$result"
}

vault_api() {
  local method=$1
  local raw_path=$2
  local ns=${3-}
  local stdout_file stderr_file rc stdout stderr code
  stdout_file=$(mktemp)
  stderr_file=$(mktemp)
  rc=0
  if [[ "$method" == "LIST" ]]; then
    if [[ -n "$ns" ]]; then
      VAULT_NAMESPACE="$ns" vault list -format=json "$raw_path" >"$stdout_file" 2>"$stderr_file" || rc=$?
    else
      VAULT_NAMESPACE= vault list -format=json "$raw_path" >"$stdout_file" 2>"$stderr_file" || rc=$?
    fi
  else
    if [[ -n "$ns" ]]; then
      VAULT_NAMESPACE="$ns" vault read -format=json "$raw_path" >"$stdout_file" 2>"$stderr_file" || rc=$?
    else
      VAULT_NAMESPACE= vault read -format=json "$raw_path" >"$stdout_file" 2>"$stderr_file" || rc=$?
    fi
  fi
  stdout=$(cat "$stdout_file")
  stderr=$(cat "$stderr_file")
  rm -f "$stdout_file" "$stderr_file"

  if [[ $rc -eq 0 ]]; then
    VAULT_API_HTTP=200
    if [[ "$method" == "LIST" ]]; then
      VAULT_API_BODY=$(printf '%s' "$stdout" | jq -c '
        if type == "array" then {data:{keys:.}}
        elif (.data.keys? != null) then {data:{keys:.data.keys, key_info:(.data.key_info // {})}}
        elif (.keys? != null) then {data:{keys:.keys, key_info:(.key_info // {})}}
        else {data:{keys:[]}}
        end
      ')
    else
      VAULT_API_BODY=$(printf '%s' "$stdout" | jq -c 'if .data then . else {data:.} end')
    fi
    return 0
  fi

  if [[ $rc -eq 2 && ( -z "${stderr// /}" || "$stderr" == *"No value found"* ) ]]; then
    VAULT_API_HTTP=404
    VAULT_API_BODY='{"errors":[]}'
    return 1
  fi

  code=$(printf '%s\n' "$stderr" | sed -n 's/.*Code: \([0-9][0-9][0-9]\).*/\1/p')
  code=${code%%$'\n'*}
  if [[ -n "$code" ]]; then
    VAULT_API_HTTP=$code
  else
    VAULT_API_HTTP=000
  fi
  VAULT_API_BODY=$(jq -n --arg err "$stderr" '{errors:[$err]}')
  return 1
}

destination_where() {
  local type=$1
  local name=$2
  local body=$3
  printf '%s' "$body" | jq -c --arg type "$type" --arg name "$name" '
    def aws_account:
      (.role_arn // .account_id // "") as $raw
      | if ($raw | test("^[0-9]{12}$")) then $raw
        elif ($raw | test("arn:aws[a-z0-9-]*:iam::[0-9]{12}:")) then
          ($raw | capture("arn:aws[a-z0-9-]*:iam::(?<id>[0-9]{12}):").id)
        else ""
        end;
    def azure_name:
      (.key_vault_uri // "") as $u
      | if ($u | test("^https://[^./]+\\.vault\\.")) then
          ($u | capture("^https://(?<n>[^./]+)\\.vault\\.").n)
        else ""
        end;
    def as_csv:
      if type == "array" then join(",")
      elif . == null then ""
      else tostring
      end;
    (.data // {}) as $d
    | ($d.connection_details // {}) as $c
    | ($d.type // $type) as $t
    | ($c.replication_locations | as_csv) as $gcp_locations
    | {
        provider: (
          if $t == "aws-sm" then "aws-secrets-manager"
          elif $t == "azure-kv" then "azure-key-vault"
          elif $t == "gcp-sm" then "gcp-secret-manager"
          elif $t == "gh" then "github"
          elif $t == "vercel-project" then "vercel"
          else $t
          end
        ),
        destination_type: $t,
        destination_name: ($d.name // $name),
        account: (if $t == "aws-sm" then ($c | aws_account) else "" end),
        region: (
          if $t == "aws-sm" then ($c.region // "")
          elif $t == "azure-kv" then ($c.region // $c.location // "")
          elif $t == "gcp-sm" then $gcp_locations
          else ($c.region // "")
          end
        ),
        role_arn: (if $t == "aws-sm" then ($c.role_arn // "") else "" end),
        key_vault: (if $t == "azure-kv" then ($c | azure_name) else "" end),
        key_vault_uri: (if $t == "azure-kv" then ($c.key_vault_uri // "") else "" end),
        tenant_id: (if $t == "azure-kv" then ($c.tenant_id // "") else "" end),
        cloud: (if $t == "azure-kv" then ($c.cloud // "") else "" end),
        project: (if $t == "gcp-sm" then ($c.project_id // "") else "" end),
        locations: (if $t == "gcp-sm" then $gcp_locations else "" end),
        repository: (if $t == "gh" then ($c.repository_name // "") else "" end),
        organization: (if $t == "gh" then ($c.organization_name // $c.repository_owner // "") else "" end),
        environment: (if $t == "gh" then ($c.environment_name // "") else "" end),
        enterprise_url: (if $t == "gh" then ($c.enterprise_url // "") else "" end),
        project_id: (if $t == "vercel-project" then ($c.project_id // "") else "" end),
        team_id: (if $t == "vercel-project" then ($c.team_id // "") else "" end),
        environments: (if $t == "vercel-project" then ($c.deployment_environments | as_csv) else "" end)
      }
  '
}

classify_sync() {
  local http=$1
  local body=$2
  local err
  if [[ "$http" =~ ^2 ]]; then
    printf 'available'
    return 0
  fi
  err=$(printf '%s' "$body" | jq -r '(.errors // []) | join(" ")' 2>/dev/null || true)
  if [[ "$http" == "404" && -z "${err// /}" ]]; then
    printf 'available'
    return 0
  fi
  printf 'unavailable'
}

collect_namespaces() {
  local parent=$1
  local http body keys child full
  printf '%s\n' "$parent" >> "$NS_FILE"
  vault_api LIST "sys/namespaces" "$parent" || true
  http=$VAULT_API_HTTP
  body=$VAULT_API_BODY
  if [[ "$http" == "404" ]]; then
    return 0
  fi
  if [[ ! "$http" =~ ^2 ]]; then
    echo "No se pudieron listar los namespaces en $(display_ns "$parent") (HTTP ${http})." >&2
    printf '%s\n' "$body" >&2
    exit 1
  fi
  keys=$(printf '%s' "$body" | jq -r '.data.keys[]? // empty')
  while IFS= read -r child; do
    [[ -n "$child" ]] || continue
    child=${child%/}
    if [[ -z "$parent" ]]; then
      full=$child
    else
      full="${parent}/${child}"
    fi
    collect_namespaces "$full"
  done <<< "$keys"
}

record_sync() {
  local ns=$1
  local http body avail type name names_http names_body assoc_http assoc_body api
  local where dest_http dest_body
  vault_api LIST "sys/sync/destinations" "$ns" || true
  http=$VAULT_API_HTTP
  body=$VAULT_API_BODY
  avail=$(classify_sync "$http" "$body")
  printf '%s\t%s\n' "$ns" "$avail" >> "$SYNC_API_FILE"
  if [[ "$avail" != "available" ]]; then
    if [[ ! -f "${TMP_DIR}/sync_warned" ]]; then
      echo "[aviso] Secrets Sync no se pudo consultar en $(display_ns "$ns") (HTTP ${http}). Esos secretos quedan como secret_sync=unknown." >&2
      printf '%s\n' "$body" >&2
      touch "${TMP_DIR}/sync_warned"
    fi
    return 0
  fi
  if [[ ! "$http" =~ ^2 ]]; then
    return 0
  fi
  while IFS= read -r type; do
    [[ -n "$type" ]] || continue
    type=${type%/}
    api="sys/sync/destinations/$(join_path "$type")"
    vault_api LIST "$api" "$ns" || true
    names_http=$VAULT_API_HTTP
    names_body=$VAULT_API_BODY
    if [[ ! "$names_http" =~ ^2 ]]; then
      continue
    fi
    while IFS= read -r name; do
      [[ -n "$name" ]] || continue
      name=${name%/}
      where='{}'
      api="sys/sync/destinations/$(join_path "$type" "$name")"
      vault_api GET "$api" "$ns" || true
      dest_http=$VAULT_API_HTTP
      dest_body=$VAULT_API_BODY
      if [[ "$dest_http" =~ ^2 ]]; then
        where=$(destination_where "$type" "$name" "$dest_body")
      else
        echo "[aviso] destino ${type}/${name} en $(display_ns "$ns") HTTP ${dest_http}" >&2
      fi
      api="sys/sync/destinations/$(join_path "$type" "$name")/associations"
      vault_api GET "$api" "$ns" || true
      assoc_http=$VAULT_API_HTTP
      assoc_body=$VAULT_API_BODY
      if [[ ! "$assoc_http" =~ ^2 ]]; then
        echo "[aviso] asociaciones ${type}/${name} en $(display_ns "$ns") HTTP ${assoc_http}" >&2
        continue
      fi
      printf '%s' "$assoc_body" | jq -c \
        --arg ns "$ns" \
        --arg dest "${type}/${name}" \
        --argjson where "$where" \
        '
          (.data.associated_secrets // {})
          | to_entries[]
          | {
              namespace: $ns,
              accessor: (.value.accessor // ""),
              secret_name: (.value.secret_name // ""),
              destination: $dest,
              sync_status: (.value.sync_status // "UNKNOWN"),
              where: $where
            }
        ' >> "$SYNC_INDEX"
    done < <(printf '%s' "$names_body" | jq -r '.data.keys[]? // empty')
  done < <(printf '%s' "$body" | jq -r '.data.keys[]? // empty')
}

sync_api_for() {
  local ns=$1
  local value
  value=$(awk -F '\t' -v ns="$ns" '$1 == ns { print $2; exit }' "$SYNC_API_FILE")
  if [[ -z "$value" ]]; then
    printf 'unavailable'
  else
    printf '%s' "$value"
  fi
}

format_secret_row() {
  local meta_json=$1
  local matches_json=$2
  local ns_col=$3
  local mount=$4
  local mount_local=$5
  local secret_path=$6
  local absolute=$7
  local sync_api=$8
  local max_age=$9
  jq -nr \
    --argjson meta "$meta_json" \
    --argjson matches "$matches_json" \
    --arg namespace "$ns_col" \
    --arg mount "$mount" \
    --arg mount_local "$mount_local" \
    --arg secret_path "$secret_path" \
    --arg absolute "$absolute" \
    --arg sync_api "$sync_api" \
    --arg max_age "$max_age" \
    '
      def ts_epoch:
        if type != "string" or . == "" then null
        else sub("\\.[0-9]+"; "") | fromdateiso8601
        end;
      def place($k; $v):
        if ($v | tostring) == "" or $v == null then empty else "\($k)=\($v)" end;
      def target:
        . as $m
        | ($m.where // {}) as $w
        | [
            ($w.provider // $m.destination),
            ($m.destination // ""),
            ("status=" + ($m.sync_status // "")),
            place("account"; $w.account),
            place("region"; $w.region),
            place("role_arn"; $w.role_arn),
            place("key_vault"; $w.key_vault),
            place("key_vault_uri"; $w.key_vault_uri),
            place("tenant"; $w.tenant_id),
            place("cloud"; $w.cloud),
            place("project"; $w.project),
            (if ($w.locations // "") != "" and ($w.locations // "") != ($w.region // "") then place("locations"; $w.locations) else empty end),
            place("organization"; $w.organization),
            place("repository"; $w.repository),
            place("environment"; $w.environment),
            place("enterprise_url"; $w.enterprise_url),
            place("project_id"; $w.project_id),
            place("team_id"; $w.team_id),
            place("environments"; $w.environments)
          ] | join(" ");

      $meta.data as $d
      | ($d.created_time // "") as $created_raw
      | (
          ($d.current_version // 0) as $cv
          | if $cv == 0 then ""
            else ($d.versions[$cv | tostring].created_time // "")
            end
        ) as $updated_raw
      | ($created_raw | ts_epoch) as $created_epoch
      | ($updated_raw | ts_epoch) as $updated_epoch
      | [$created_epoch, $updated_epoch] | map(select(. != null)) as $stamps
      | (if ($stamps | length) == 0 then null else ($stamps | max) end) as $last
      | (now | floor) as $now
      | (if $last == null then "" else (($now - $last) | floor | tostring) end) as $age_seconds
      | (if $last == null then "" else ((($now - $last) / 86400) | floor | tostring) end) as $age_days
      | (
          if $last == null then "unknown"
          elif ($now - $last) > (($max_age | tonumber) * 86400) then "true"
          else "false"
          end
        ) as $stale
      | ($d.custom_metadata // {}) as $cm
      | (if ($cm | type) != "object" or ($cm | length) == 0 then "true" else "false" end) as $missing
      | (
          if $sync_api != "available" then "unknown"
          elif ($matches | length) > 0 then "true"
          else "false"
          end
        ) as $synced
      | ([$matches[]? | "\(.destination):\(.sync_status)"] | join("|")) as $dests
      | ([$matches[]? | target] | join("|")) as $where_text
      | ([$matches[]? | {
            destination,
            sync_status,
            provider: (.where.provider // ""),
            account: (.where.account // ""),
            region: (.where.region // ""),
            role_arn: (.where.role_arn // ""),
            key_vault: (.where.key_vault // ""),
            key_vault_uri: (.where.key_vault_uri // ""),
            tenant_id: (.where.tenant_id // ""),
            cloud: (.where.cloud // ""),
            project: (.where.project // ""),
            locations: (.where.locations // ""),
            organization: (.where.organization // ""),
            repository: (.where.repository // ""),
            environment: (.where.environment // ""),
            enterprise_url: (.where.enterprise_url // ""),
            project_id: (.where.project_id // ""),
            team_id: (.where.team_id // ""),
            environments: (.where.environments // "")
          }] | tojson) as $targets
      | {
          csv: ([
            $namespace,
            $mount,
            $mount_local,
            $secret_path,
            $absolute,
            $created_raw,
            (if $updated_raw == "" then $created_raw else $updated_raw end),
            $age_seconds,
            $age_days,
            $max_age,
            $stale,
            $missing,
            ($cm.owner // ""),
            ($cm.app // ""),
            ($cm.project // ""),
            ($cm.platform // ""),
            ($cm.env // ""),
            ($cm.team // ""),
            ($cm | tojson),
            $synced,
            $dests,
            $where_text,
            $targets,
            $sync_api
          ] | @csv),
          stale: $stale,
          missing: $missing,
          synced: $synced
        }
    '
}

emit_secret() {
  local ns=$1
  local mount=$2
  local secret_path=$3
  local accessor=$4
  local mount_local=$5
  local -a segments parts
  local part api http body ns_col absolute matches sync_api parsed
  segments=("$mount" "metadata")
  IFS='/' read -ra parts <<< "$secret_path"
  for part in "${parts[@]}"; do
    [[ -n "$part" ]] && segments+=("$part")
  done
  api=$(join_path "${segments[@]}")
  vault_api GET "$api" "$ns" || true
  http=$VAULT_API_HTTP
  body=$VAULT_API_BODY
  ns_col=$(display_ns "$ns")
  if [[ "$ns_col" == "/" ]]; then
    absolute="/${mount}/${secret_path}"
  else
    absolute="${ns_col}/${mount}/${secret_path}"
  fi
  if [[ ! "$http" =~ ^2 ]]; then
    echo "[aviso] sin metadata para ${absolute} (HTTP ${http})" >&2
    return 0
  fi
  if ! printf '%s' "$body" | jq -e '.data' >/dev/null 2>&1; then
    echo "[aviso] respuesta de metadata no válida para ${absolute}" >&2
    return 0
  fi
  matches=$(jq -c -s \
    --arg ns "$ns" \
    --arg acc "$accessor" \
    --arg name "$secret_path" \
    'map(select(.namespace == $ns and .secret_name == $name and (.accessor == $acc or $acc == "" or .accessor == "")))' \
    "$SYNC_INDEX")
  sync_api=$(sync_api_for "$ns")
  parsed=$(format_secret_row "$body" "$matches" "$ns_col" "$mount" "$mount_local" "$secret_path" "$absolute" "$sync_api" "$MAX_AGE_DAYS")
  printf '%s\n' "$(printf '%s' "$parsed" | jq -r '.csv')" >> "$CSV_TMP"
  TOTAL=$((TOTAL + 1))
  case "$(printf '%s' "$parsed" | jq -r '.missing')" in
    true) MISSING=$((MISSING + 1)) ;;
  esac
  case "$(printf '%s' "$parsed" | jq -r '.stale')" in
    true) STALE=$((STALE + 1)) ;;
  esac
  case "$(printf '%s' "$parsed" | jq -r '.synced')" in
    true) SYNCED=$((SYNCED + 1)) ;;
    unknown) SYNC_UNKNOWN=$((SYNC_UNKNOWN + 1)) ;;
  esac
}

walk_kv() {
  local ns=$1
  local mount=$2
  local prefix=$3
  local accessor=$4
  local mount_local=$5
  local -a segments parts
  local part api http body key
  segments=("$mount" "metadata")
  if [[ -n "$prefix" ]]; then
    IFS='/' read -ra parts <<< "${prefix%/}"
    for part in "${parts[@]}"; do
      [[ -n "$part" ]] && segments+=("$part")
    done
  fi
  api=$(join_path "${segments[@]}")
  vault_api LIST "$api" "$ns" || true
  http=$VAULT_API_HTTP
  body=$VAULT_API_BODY
  if [[ "$http" == "404" ]]; then
    return 0
  fi
  if [[ ! "$http" =~ ^2 ]]; then
    echo "[aviso] no se pudo listar $(display_ns "$ns")/${mount}/${prefix} (HTTP ${http})" >&2
    return 0
  fi
  while IFS= read -r key; do
    [[ -n "$key" ]] || continue
    if [[ "$key" == */ ]]; then
      walk_kv "$ns" "$mount" "${prefix}${key}" "$accessor" "$mount_local"
    else
      emit_secret "$ns" "$mount" "${prefix}${key}" "$accessor" "$mount_local"
    fi
  done < <(printf '%s' "$body" | jq -r '.data.keys[]? // empty')
}

scan_namespace() {
  local ns=$1
  local http body mount accessor mount_local
  echo "[scan] $(display_ns "$ns")"
  record_sync "$ns"
  vault_api GET "sys/mounts" "$ns" || true
  http=$VAULT_API_HTTP
  body=$VAULT_API_BODY
  if [[ ! "$http" =~ ^2 ]]; then
    echo "[aviso] no se pudieron listar los mounts de $(display_ns "$ns") (HTTP ${http})" >&2
    return 0
  fi
  while IFS=$'\t' read -r mount accessor mount_local; do
    [[ -n "$mount" ]] || continue
    mount=${mount%/}
    walk_kv "$ns" "$mount" "" "$accessor" "$mount_local"
  done < <(printf '%s' "$body" | jq -r '
    .data
    | to_entries[]
    | select(.value.type == "kv")
    | select((((.value.options // {}).version) // "1" | tostring) == "2")
    | "\(.key)\t\(.value.accessor // "")\t\((.value.local // false) | tostring)"
  ')
}

main() {
  local ns out_dir
  parse_args "$@"
  prepare_workspace
  : > "$NS_FILE"
  collect_namespaces ""
  printf '%s\n' \
    "namespace,secret_mount,mount_local,secret_path,absolute_path,created_time,updated_time,age_seconds,age_days,max_age_days,older_than_threshold,missing_metadata,owner,app,project,platform,env,team,custom_metadata,secret_sync,sync_destinations,sync_where,sync_targets,sync_api" \
    > "$CSV_TMP"
  while IFS= read -r ns; do
    scan_namespace "$ns"
  done < "$NS_FILE"
  out_dir=$(dirname "$OUTPUT")
  mkdir -p "$out_dir"
  mv "$CSV_TMP" "$OUTPUT"
  # El trap borra TMP_DIR. El CSV ya está fuera.
  echo
  echo "secretos=${TOTAL}"
  echo "missing_metadata=${MISSING}"
  echo "secret_sync=${SYNCED}"
  echo "secret_sync_unknown=${SYNC_UNKNOWN}"
  echo "older_than_threshold=${STALE}"
  echo "max_age_days=${MAX_AGE_DAYS}"
  echo "csv=${OUTPUT}"
}

if [[ "${BASH_SOURCE[0]}" == "$0" ]]; then
  main "$@"
fi
INVENTORY_SCRIPT
chmod +x scripts/vault_secret_inventory.sh
bash -n scripts/vault_secret_inventory.sh
echo "script=$(pwd)/scripts/vault_secret_inventory.sh"


## Lectura del CSV

La celda carga `/tmp/vault/secret-inventory-90d.csv` con pandas. La vista incluye el namespace raíz `/` y el árbol `/ns1` … `/ns5`. La primera tabla cuenta los flags, la segunda resume cada namespace y la tercera es el inventario, de 25 secretos por página. Los secretos sin metadatos quedan arriba.

Las columnas de metadatos salen de `custom_metadata`: cada clave del JSON es una columna. No hay una lista fija de campos.

Con 90 días, los secretos recién escritos tienen que salir con `older_than_threshold=false`. `missing_metadata=true` tiene que ser 25. `secret_sync` tiene que ser `false` si la API de sync responde y no hay destinos, o `unknown` si ese endpoint no está disponible en el cluster.


In [ ]:
%%bash
#set -euo pipefail
#set -a
#source /tmp/vault/config.env
#set +a
#unset VAULT_NAMESPACE
SECRET_MAX_AGE_DAYS=90 ./scripts/vault_secret_inventory.sh --output /tmp/vault/secret-inventory-90d.csv


In [ ]:
import json

import ipywidgets as widgets
import pandas as pd
from IPython.display import display

PAGE_SIZE = 25
pd.set_option("display.max_rows", PAGE_SIZE)
pd.set_option("display.max_columns", None)
pd.set_option("display.max_colwidth", 140)

IDENTITY_COLUMNS = [
    "namespace",
    "secret_mount",
    "mount_local",
    "secret_path",
    "absolute_path",
    "missing_metadata",
]
TAIL_COLUMNS = [
    "secret_sync",
    "sync_where",
    "sync_api",
    "older_than_threshold",
    "age_seconds",
    "age_days",
    "created_time",
    "updated_time",
]

def parse_metadata(value):
    if value is None or (isinstance(value, float) and pd.isna(value)):
        return {}
    text = str(value).strip()
    if text in ("", "{}", "null", "None"):
        return {}
    try:
        parsed = json.loads(text)
    except json.JSONDecodeError:
        return {}
    if not isinstance(parsed, dict):
        return {}
    return {str(key): "" if item is None else str(item) for key, item in parsed.items()}

def expand_metadata(frame):
    if "custom_metadata" not in frame.columns:
        return pd.DataFrame(index=frame.index)
    records = frame["custom_metadata"].map(parse_metadata)
    keys = sorted({key for record in records for key in record})
    return pd.DataFrame(
        [{key: record.get(key, "") for key in keys} for record in records],
        index=frame.index,
        columns=keys,
    )

def as_bool(series):
    if series.dtype == bool:
        return series
    return series.astype("string").str.lower().map({"true": True, "false": False})

def es_lote(series):
    return series.astype(str).str.match(r"^/ns[1-5](?:/|$)")

def es_vista(series):
    text = series.astype(str)
    return text.eq("/") | es_lote(text)

def load_demo(path):
    frame = pd.read_csv(path)
    demo = frame[es_vista(frame["namespace"])].copy()
    for column in ("missing_metadata", "older_than_threshold", "mount_local"):
        demo[column] = as_bool(demo[column])
    for column in ("age_seconds", "age_days", "max_age_days"):
        if column in demo.columns:
            demo[column] = pd.to_numeric(demo[column], errors="coerce")
    return frame, demo

def conteos(frame):
    pieces = []
    for column in ("missing_metadata", "mount_local", "secret_sync", "sync_api", "older_than_threshold"):
        counts = frame[column].value_counts(dropna=False)
        piece = counts.rename_axis("valor").reset_index(name="secretos")
        piece.insert(0, "campo", column)
        pieces.append(piece)
    return pd.concat(pieces, ignore_index=True)

def por_namespace(frame):
    grouped = (
        frame.groupby("namespace", as_index=False)
        .agg(
            secretos=("absolute_path", "size"),
            mounts=("secret_mount", "nunique"),
            secretos_mount_local=("mount_local", "sum"),
            sin_metadatos=("missing_metadata", "sum"),
            sincronizados=("secret_sync", lambda values: int((values == "true").sum())),
            mas_viejos=("older_than_threshold", "sum"),
        )
        .sort_values("namespace")
        .reset_index(drop=True)
    )
    grouped["sin_metadatos"] = grouped["sin_metadatos"].astype(int)
    grouped["secretos_mount_local"] = grouped["secretos_mount_local"].astype(int)
    grouped["mas_viejos"] = grouped["mas_viejos"].astype(int)
    return grouped

cluster, secrets = load_demo("/tmp/vault/secret-inventory-90d.csv")
lote = secrets[es_lote(secrets["namespace"])].copy()
metadata = expand_metadata(secrets)
vista = (
    pd.concat(
        [
            secrets[[column for column in IDENTITY_COLUMNS if column in secrets.columns]],
            metadata,
            secrets[[column for column in TAIL_COLUMNS if column in secrets.columns]],
        ],
        axis=1,
    )
    .sort_values(
        ["missing_metadata", "namespace", "secret_mount", "secret_path"],
        ascending=[False, True, True, True],
    )
    .reset_index(drop=True)
)

def tabla_paginada(frame):
    total = len(frame)
    pages = max(1, (total + PAGE_SIZE - 1) // PAGE_SIZE)
    page_w = widgets.BoundedIntText(
        value=1,
        min=1,
        max=pages,
        description="Página",
        style={"description_width": "initial"},
    )
    info = widgets.HTML()
    out = widgets.Output()

    def draw(change=None):
        page = change["new"] if isinstance(change, dict) and "new" in change else page_w.value
        start = (page - 1) * PAGE_SIZE
        chunk = frame.iloc[start:start + PAGE_SIZE]
        end = start + len(chunk)
        info.value = "0 secretos" if total == 0 else f"{start + 1}–{end} de {total}"
        out.clear_output(wait=True)
        if total == 0:
            return
        with out:
            display(chunk)

    page_w.observe(draw, names="value")
    display(widgets.VBox([page_w, info, out]))
    draw()

display(conteos(secrets))
display(por_namespace(secrets))
tabla_paginada(vista)

assert "/" in set(secrets["namespace"].astype(str))
assert len(lote) == 100, len(lote)
assert lote["namespace"].nunique() == 15
assert int(lote["missing_metadata"].sum()) == 25
assert int((~lote["older_than_threshold"]).sum()) == 100
if (lote["secret_sync"] == "true").any():
    raise SystemExit("hay secretos del lote con Secrets Sync y este notebook no crea destinos")
meta_lote = expand_metadata(lote)
for column in meta_lote.columns:
    assert column in vista.columns, column
assert set(meta_lote.columns) >= {"owner", "app", "project", "platform", "env", "team"}


## Exploración

Esta celda usa el dataframe `vista` de la celda anterior. Hay que ejecutar esa antes. No vuelve a leer Vault.

Namespace y mount se filtran en la tabla: al acotar un namespace, el desplegable de mount solo ofrece los de ese namespace, y el tablero y las cifras siguen el mismo recorte. El tablero muestra cada namespace y, dentro, un mount por fila. El color de cada tramo es el valor de Agrupar. Si hay más de 12 valores, el resto se agrupa como `otros`. Los vacíos salen como `(vacío)`. La tabla arranca sin metadatos hasta pulsar **Mostrar**. En cada columna se puede filtrar, ordenar con el nombre y cambiar la anchura arrastrando el borde. Al pasar el ratón por la cabecera aparece una ×: al pulsarla, la columna se oculta. Debajo de la tabla, cada columna oculta tiene un botón para recuperarla. La paginación queda debajo de la tabla.


In [11]:
import html as html_lib

import anywidget
import ipywidgets as widgets
import traitlets

if "vista" not in globals():
    raise RuntimeError("Ejecuta antes la celda que construye vista. Esta celda no lee Vault.")

PAGE_SIZE = globals().get("PAGE_SIZE", 25)
PALETTE = [
    "#4f46e5", "#0f766e", "#c2410c", "#0369a1", "#7c3aed", "#b45309",
    "#be123c", "#047857", "#4338ca", "#a16207", "#0e7490", "#9f1239",
]

def etiqueta_grafico(value):
    text = "" if value is None else str(value).strip()
    if text in ("", "nan", "None", "<NA>"):
        return "(vacío)"
    if text == "/":
        return "(root)"
    if text.startswith("/"):
        text = text[1:]
    return text.replace("/", " > ")

def mapa_color(dimension):
    nombres = sorted({etiqueta_grafico(valor) for valor in vista[dimension].tolist()})
    colores = {nombre: PALETTE[i % len(PALETTE)] for i, nombre in enumerate(nombres)}
    colores["otros"] = "#94a3b8"
    return colores

def limitar_grupos(serie, limite=12):
    conteo = serie.value_counts()
    if len(conteo) <= limite:
        return serie
    mantener = set(conteo.head(limite - 1).index)
    return serie.where(serie.isin(mantener), "otros")

def es_si(valor):
    if valor is True:
        return True
    if valor is False:
        return False
    text = "" if valor is None else str(valor).strip().lower()
    if text in ("true", "sí", "si", "yes", "1"):
        return True
    if text in ("false", "no", "0"):
        return False
    return None

def pastilla(texto, clase=""):
    return f'<span class="inv-pill {clase}">{texto}</span>'

def celda(columna, valor):
    flag = es_si(valor)
    if columna == "secret_sync":
        if flag is True:
            return pastilla("sí", "inv-pill-ok")
        if flag is False:
            return pastilla("no")
        text = "" if valor is None else str(valor).strip().lower()
        if text in ("", "nan", "none", "<na>"):
            return '<span class="inv-muted">—</span>'
        if text == "unknown":
            return pastilla("unknown", "inv-pill-warn")
        return pastilla(html_lib.escape(str(valor).strip()), "inv-pill-warn")
    if columna == "missing_metadata":
        if flag is True:
            return pastilla("sí", "inv-pill-bad")
        if flag is False:
            return pastilla("no")
    if columna == "older_than_threshold":
        if flag is True:
            return pastilla("sí", "inv-pill-warn")
        if flag is False:
            return pastilla("no", "inv-pill-ok")
    if flag is True:
        return pastilla("sí", "inv-pill-ok")
    if flag is False:
        return pastilla("no")
    text = "" if valor is None else str(valor)
    if text in ("", "nan", "None", "<NA>"):
        return '<span class="inv-muted">—</span>'
    return html_lib.escape(text)

def columnas_tabla():
    ident = [column for column in IDENTITY_COLUMNS if column in vista.columns]
    tail = [column for column in TAIL_COLUMNS if column in vista.columns]
    elegido = campo_w.value
    if not meta_btn.value:
        extra = []
    elif elegido == "todos":
        extra = list(metadata_columns)
    elif elegido in metadata_columns:
        extra = [elegido]
    else:
        extra = []
    return ident + extra + tail

def texto_celda(columna, valor):
    flag = es_si(valor)
    if columna == "secret_sync":
        if flag is True:
            return "sí"
        if flag is False:
            return "no"
        text = "" if valor is None else str(valor).strip()
        if text.lower() in ("", "nan", "none", "<na>"):
            return ""
        return text
    if flag is True:
        return "sí"
    if flag is False:
        return "no"
    text = "" if valor is None else str(valor).strip()
    if text.lower() in ("", "nan", "none", "<na>"):
        return ""
    return text

class TablaInventario(anywidget.AnyWidget):
    _esm = """
function escapeHtml(value) {
  return String(value ?? "")
    .replaceAll("&", "&amp;")
    .replaceAll("<", "&lt;")
    .replaceAll(">", "&gt;")
    .replaceAll('"', "&quot;");
}
function pill(text, kind) {
  const safe = escapeHtml(text);
  if (!safe) return '<span class="inv-muted">—</span>';
  if (!kind) return safe;
  return `<span class="inv-pill ${kind}">${safe}</span>`;
}
function paintCell(column, text) {
  if (column === "secret_sync") {
    if (text === "sí") return pill(text, "inv-pill-ok");
    if (text === "no") return pill(text, "");
    if (text) return pill(text, "inv-pill-warn");
  }
  if (column === "missing_metadata") {
    if (text === "sí") return pill(text, "inv-pill-bad");
    if (text === "no") return pill(text, "");
  }
  if (column === "older_than_threshold") {
    if (text === "no") return pill(text, "inv-pill-ok");
    if (text === "sí") return pill(text, "inv-pill-warn");
  }
  if (text === "sí") return pill(text, "inv-pill-ok");
  if (text === "no") return pill(text, "");
  return pill(text, "");
}
function render({ model, el }) {
  const state = { sortCol: null, sortDir: 1, filters: {}, exact: {}, widths: {}, page: 1, colsKey: null, hidden: {} };
  el.innerHTML = '<div class="inv-table-wrap"><table class="inv-table inv-grid"><thead></thead><tbody></tbody></table></div><div class="inv-pager"></div><div class="inv-restore" hidden></div>';
  const thead = el.querySelector("thead");
  const tbody = el.querySelector("tbody");
  const pager = el.querySelector(".inv-pager");
  const restore = el.querySelector(".inv-restore");
  const menu = document.createElement("div");
  menu.className = "inv-menu";
  menu.hidden = true;
  el.append(menu);
  function columns() { return model.get("columns") || []; }
  function shown() {
    return columns().map((col, index) => ({ col, index })).filter((item) => !state.hidden[item.col]);
  }
  function allRows() { return model.get("rows") || []; }
  function pageSize() { return model.get("page_size") || 25; }
  function cellText(row, index) { return String(row[index] ?? ""); }
  function passes(row, index, col) {
    const query = (state.filters[col] || "").trim().toLowerCase();
    const text = cellText(row, index);
    if (state.exact[col]) return query ? text.toLowerCase() === query : !text;
    return !query || text.toLowerCase().includes(query);
  }
  function matched() {
    const cols = columns();
    let data = allRows().filter((row) => cols.every((col, index) => passes(row, index, col)));
    if (state.sortCol !== null && state.sortCol < cols.length) {
      const index = state.sortCol;
      const dir = state.sortDir;
      data = data.slice().sort((a, b) => cellText(a, index).localeCompare(cellText(b, index), "es", { numeric: true, sensitivity: "base" }) * dir);
    }
    return data;
  }
  function suggestions(index, query) {
    const cols = columns();
    const pool = allRows().filter((row) => cols.every((col, i) => i === index || passes(row, i, col)));
    const counts = new Map();
    pool.forEach((row) => {
      const text = cellText(row, index);
      const label = text || "—";
      counts.set(label, (counts.get(label) || 0) + 1);
    });
    const q = query.trim().toLowerCase();
    return [...counts.entries()]
      .filter(([label]) => !q || label.toLowerCase().includes(q))
      .sort((a, b) => b[1] - a[1] || a[0].localeCompare(b[0], "es", { numeric: true }))
      .slice(0, 8);
  }
  function hideMenu() { menu.hidden = true; }
  const inputs = {};
  function publishScope() {
    const filters = {};
    const exact = {};
    ["namespace", "secret_mount"].forEach((col) => {
      filters[col] = state.filters[col] || "";
      exact[col] = !!state.exact[col];
    });
    model.set("column_filters", filters);
    model.set("column_exact", exact);
    model.save_changes();
  }
  function clearMountIfOrphan() {
    const cols = columns();
    const ni = cols.indexOf("namespace");
    const mi = cols.indexOf("secret_mount");
    if (ni < 0 || mi < 0) return;
    const query = (state.filters.secret_mount || "").trim();
    if (!query && !state.exact.secret_mount) return;
    const alive = allRows().some((row) => passes(row, ni, "namespace") && passes(row, mi, "secret_mount"));
    if (!alive) {
      state.filters.secret_mount = "";
      state.exact.secret_mount = false;
      if (inputs.secret_mount) inputs.secret_mount.value = "";
    }
  }
  function aplicarVista(col) {
    if (col === "namespace") clearMountIfOrphan();
    if (col === "namespace" || col === "secret_mount") publishScope();
  }
  function showMenu(input, index, col) {
    const options = suggestions(index, input.value);
    menu.replaceChildren();
    if (!options.length) { hideMenu(); return; }
    options.forEach(([label, count]) => {
      const item = document.createElement("button");
      item.type = "button";
      item.innerHTML = `<span>${escapeHtml(label)}</span><b>${count}</b>`;
      item.addEventListener("mousedown", (ev) => {
        ev.preventDefault();
        input.value = label === "—" ? "" : label;
        state.filters[col] = label === "—" ? "" : label;
        state.exact[col] = true;
        state.page = 1;
        hideMenu();
        aplicarVista(col);
        paintBody();
      });
      menu.append(item);
    });
    const rect = input.getBoundingClientRect();
    menu.style.left = rect.left + "px";
    menu.style.top = (rect.bottom + 4) + "px";
    menu.style.width = Math.max(rect.width, 180) + "px";
    menu.hidden = false;
  }
  function hideColumn(col) {
    state.hidden[col] = true;
    state.filters[col] = "";
    state.exact[col] = false;
    if (inputs[col]) inputs[col].value = "";
    if (state.sortCol !== null && columns()[state.sortCol] === col) state.sortCol = null;
    state.page = 1;
    aplicarVista(col);
    buildHead();
    paintBody();
  }
  function showColumn(col) {
    delete state.hidden[col];
    state.page = 1;
    buildHead();
    paintBody();
  }
  function buildHead() {
    const cols = columns();
    state.colsKey = cols.join("\u0000");
    thead.replaceChildren();
    const row = document.createElement("tr");
    shown().forEach(({ col, index }) => {
      const th = document.createElement("th");
      th.style.width = (state.widths[col] || 160) + "px";
      const button = document.createElement("button");
      button.type = "button";
      button.className = "inv-sort";
      button.addEventListener("click", () => {
        if (state.sortCol === index) state.sortDir *= -1;
        else { state.sortCol = index; state.sortDir = 1; }
        state.page = 1;
        paintBody();
      });
      const input = document.createElement("input");
      input.className = "inv-filter";
      input.placeholder = "Filtrar";
      input.value = state.filters[col] || "";
      inputs[col] = input;
      input.addEventListener("input", () => {
        state.filters[col] = input.value;
        state.exact[col] = false;
        state.page = 1;
        showMenu(input, index, col);
        aplicarVista(col);
        paintBody();
      });
      input.addEventListener("focus", () => showMenu(input, index, col));
      input.addEventListener("blur", () => setTimeout(hideMenu, 120));
      const handle = document.createElement("span");
      handle.className = "inv-resize";
      handle.addEventListener("mousedown", (ev) => {
        ev.preventDefault();
        ev.stopPropagation();
        const startX = ev.clientX;
        const startW = th.getBoundingClientRect().width;
        function move(e) {
          const width = Math.max(88, startW + e.clientX - startX);
          state.widths[col] = width;
          th.style.width = width + "px";
        }
        function up() {
          window.removeEventListener("mousemove", move);
          window.removeEventListener("mouseup", up);
        }
        window.addEventListener("mousemove", move);
        window.addEventListener("mouseup", up);
      });
      const close = document.createElement("button");
      close.type = "button";
      close.className = "inv-col-x";
      close.textContent = "\u00d7";
      close.title = "Quitar columna";
      close.addEventListener("mousedown", (ev) => ev.stopPropagation());
      close.addEventListener("click", (ev) => {
        ev.preventDefault();
        ev.stopPropagation();
        hideColumn(col);
      });
      th.append(button, close, input, handle);
      row.append(th);
    });
    thead.append(row);
  }
  function paintBody() {
    const cols = shown();
    const buttons = [...thead.querySelectorAll(".inv-sort")];
    cols.forEach(({ col, index }, position) => {
      const mark = state.sortCol === index ? (state.sortDir > 0 ? " ↑" : " ↓") : "";
      if (buttons[position]) buttons[position].textContent = col + mark;
    });
    const data = matched();
    const size = pageSize();
    const pages = Math.max(1, Math.ceil(data.length / size));
    state.page = Math.min(state.page, pages);
    const start = (state.page - 1) * size;
    const slice = data.slice(start, start + size);
    tbody.replaceChildren();
    if (!cols.length) {
      const tr = document.createElement("tr");
      const td = document.createElement("td");
      td.className = "inv-empty";
      td.textContent = "Todas las columnas están ocultas";
      tr.append(td);
      tbody.append(tr);
    } else if (!allRows().length) {
      const tr = document.createElement("tr");
      const td = document.createElement("td");
      td.colSpan = Math.max(cols.length, 1);
      td.className = "inv-empty";
      td.textContent = "Ningún secreto con estos filtros";
      tr.append(td);
      tbody.append(tr);
    } else if (!slice.length) {
      const tr = document.createElement("tr");
      const td = document.createElement("td");
      td.colSpan = Math.max(cols.length, 1);
      td.className = "inv-empty";
      td.textContent = "Ningún secreto coincide con el filtro de columna";
      tr.append(td);
      tbody.append(tr);
    } else {
      slice.forEach((record) => {
        const tr = document.createElement("tr");
        cols.forEach(({ col, index }) => {
          const td = document.createElement("td");
          td.innerHTML = paintCell(col, record[index] ?? "");
          tr.append(td);
        });
        tbody.append(tr);
      });
    }
    const end = Math.min(start + size, data.length);
    pager.replaceChildren();
    const label = document.createElement("span");
    label.textContent = data.length ? `${start + 1}–${end} de ${data.length}` : "0 secretos";
    const prev = document.createElement("button");
    prev.type = "button";
    prev.textContent = "Anterior";
    prev.disabled = state.page <= 1;
    prev.addEventListener("click", () => { state.page -= 1; paintBody(); });
    const next = document.createElement("button");
    next.type = "button";
    next.textContent = "Siguiente";
    next.disabled = state.page >= pages;
    next.addEventListener("click", () => { state.page += 1; paintBody(); });
    pager.append(label, prev, next);
    const hiddenCols = columns().filter((col) => state.hidden[col]);
    restore.replaceChildren();
    restore.hidden = hiddenCols.length === 0;
    hiddenCols.forEach((col) => {
      const btn = document.createElement("button");
      btn.type = "button";
      btn.textContent = "Recuperar " + col;
      btn.addEventListener("click", () => showColumn(col));
      restore.append(btn);
    });
  }
  function paint() {
    const key = columns().join("\u0000");
    if (key !== state.colsKey) buildHead();
    paintBody();
  }
  model.on("change:rows", () => { state.page = 1; paint(); });
  model.on("change:columns", () => { state.page = 1; paint(); });
  paint();
}
export default { render };
"""
    columns = traitlets.List([]).tag(sync=True)
    rows = traitlets.List([]).tag(sync=True)
    page_size = traitlets.Int(25).tag(sync=True)
    column_filters = traitlets.Dict({}).tag(sync=True)
    column_exact = traitlets.Dict({}).tag(sync=True)

def publicar_tabla(frame):
    cols = columnas_tabla()
    filas = []
    if cols and not frame.empty:
        for rec in frame[cols].itertuples(index=False, name=None):
            filas.append([texto_celda(col, valor) for col, valor in zip(cols, rec)])
    tabla.columns = cols
    tabla.rows = filas

def html_tablero(frame, dimension):
    if frame.empty:
        return '<div class="inv-empty">Ningún secreto con estos filtros</div>'
    work = frame.copy()
    work["ns"] = work["namespace"].map(etiqueta_grafico)
    work["mount"] = work["secret_mount"].map(etiqueta_grafico)
    work["grupo"] = limitar_grupos(work[dimension].map(etiqueta_grafico))
    colores = mapa_color(dimension)
    conteo = work["grupo"].value_counts()
    legend = "".join(
        '<span class="inv-key"><i style="background:{color}"></i>{nombre}<b>{n}</b></span>'.format(
            color=colores.get(nombre, "#94a3b8"),
            nombre=html_lib.escape(str(nombre)),
            n=int(n),
        )
        for nombre, n in conteo.items()
    )
    cards = []
    max_mount = int(work.groupby(["ns", "mount"]).size().max())
    for ns, ns_n in work.groupby("ns").size().sort_values(ascending=False).items():
        block = work[work["ns"] == ns]
        mounts = []
        for mount, mcount in block.groupby("mount").size().sort_values(ascending=False).items():
            partes = block.loc[block["mount"] == mount, "grupo"].value_counts()
            segs = []
            for nombre, n in partes.items():
                pct = 100 * n / mcount
                tip = html_lib.escape(f"{nombre} · {int(n)}", quote=True)
                segs.append(
                    f'<span class="inv-seg" style="width:{pct:.4f}%;background:{colores.get(nombre, "#94a3b8")}" data-tip="{tip}"></span>'
                )
            ancho = 100 * int(mcount) / max_mount
            mounts.append(
                '<div class="inv-mount">'
                f'<div class="inv-mount-name">{html_lib.escape(str(mount))}</div>'
                f'<div class="inv-stack" style="width:{ancho:.4f}%">{"".join(segs)}</div>'
                f'<div class="inv-mount-n">{int(mcount)}</div>'
                "</div>"
            )
        cards.append(
            '<article class="inv-ns"><header><h3>'
            + html_lib.escape(str(ns))
            + f'</h3><span>{int(ns_n)}</span></header>'
            + "".join(mounts)
            + "</article>"
        )
    return (
        f'<div class="inv-legend"><span class="inv-legend-title">{html_lib.escape(str(dimension))}</span>{legend}</div>'
        '<div class="inv-board">' + "".join(cards) + "</div>"
    )

known_columns = set(IDENTITY_COLUMNS) | set(TAIL_COLUMNS)
metadata_columns = [column for column in vista.columns if column not in known_columns]
group_options = ["namespace", "secret_mount", *metadata_columns]
group_w = widgets.Dropdown(options=group_options, value="owner" if "owner" in group_options else group_options[0], description="")
campo_w = widgets.Dropdown(
    options=[("Todos", "todos"), *[(column, column) for column in metadata_columns]] or [("Todos", "todos")],
    value="owner" if "owner" in metadata_columns else "todos",
    description="",
    disabled=True,
)
meta_btn = widgets.ToggleButton(
    value=False,
    description="Mostrar",
    icon="eye-slash",
    tooltip="Muestra u oculta el metadato en la tabla",
    layout=widgets.Layout(width="100%", margin="0 0 8px 0"),
)

def al_pulsar_metadatos(change):
    activo = bool(change["new"])
    meta_btn.description = "Ocultar" if activo else "Mostrar"
    meta_btn.icon = "eye" if activo else "eye-slash"
    meta_btn.button_style = "success" if activo else ""
    campo_w.disabled = not activo

meta_btn.observe(al_pulsar_metadatos, names="value")
def filtrar():
    return vista.copy().reset_index(drop=True)

def aplicar_tabla(frame):
    filtros = dict(getattr(tabla, "column_filters", {}) or {})
    exactos = dict(getattr(tabla, "column_exact", {}) or {})
    for columna in ("namespace", "secret_mount"):
        if columna not in frame.columns:
            continue
        query = str(filtros.get(columna) or "").strip()
        exacto = bool(exactos.get(columna))
        if not query and not exacto:
            continue
        textos = frame[columna].map(lambda valor, col=columna: texto_celda(col, valor))
        if exacto:
            frame = frame[textos.eq("") if not query else textos.str.lower().eq(query.lower())]
        else:
            frame = frame[textos.str.lower().str.contains(query.lower(), regex=False, na=False)]
    return frame.reset_index(drop=True)

def redraw(change=None):
    base = filtrar()
    if change is not None and change.get("name") in ("column_filters", "column_exact"):
        recorte = aplicar_tabla(base)
        pintar_kpis(recorte)
        tablero.value = html_tablero(recorte, group_w.value)
        return
    publicar_tabla(base)
    recorte = aplicar_tabla(base)
    pintar_kpis(recorte)
    tablero.value = html_tablero(recorte, group_w.value)

def pintar_kpis(frame):
    total = len(frame)
    sin = int((frame["missing_metadata"] == True).sum()) if total else 0
    viejos = int((frame["older_than_threshold"] == True).sum()) if total else 0
    n_ns = int(frame["namespace"].nunique()) if total else 0
    n_mount = int(frame["secret_mount"].nunique()) if total else 0
    synced = int(frame["secret_sync"].map(es_si).eq(True).sum()) if total and "secret_sync" in frame.columns else 0

    def card(numero, etiqueta, clase=""):
        return f'<div class="inv-kpi {clase}"><div class="n">{numero}</div><div class="l">{etiqueta}</div></div>'

    kpis.value = '<div class="inv-kpis">' + "".join([
        card(total, "Secretos", "accent"),
        card(sin, "Sin metadatos", "bad" if sin else ""),
        card(viejos, "Sobre la edad", "warn" if viejos else ""),
        card(synced, "Con sync", "ok" if synced else ""),
        card(n_ns, "Namespaces"),
        card(n_mount, "Mounts"),
    ]) + "</div>"

def bloque(titulo, hijo, extra=""):
    cabeza = widgets.HTML(f'<div class="inv-label">{titulo}</div>')
    margen = {"margin": extra} if extra else {}
    return widgets.VBox([cabeza, hijo], layout=widgets.Layout(flex="1", min_width="160px", **margen))

for campo in (group_w, campo_w):
    campo.layout = widgets.Layout(width="100%")
    campo.add_class("inv-input")

css_w = widgets.HTML("""
<style>
.inv-shell {
  --ink: #111827;
  --muted: #6b7280;
  --line: #e6e8ee;
  --bg: #f3f4f6;
  --card: #ffffff;
  --accent: #4f46e5;
  font-family: ui-sans-serif, system-ui, -apple-system, "Segoe UI", sans-serif;
  background: var(--bg);
  border-radius: 22px;
  padding: 18px;
  color: var(--ink);
}
.inv-head { padding: 2px 4px 14px; }
.inv-head h2 { margin: 0; font-size: 20px; font-weight: 600; letter-spacing: -0.03em; }
.inv-head p { margin: 6px 0 0; font-size: 13px; line-height: 1.45; color: var(--muted); }
.inv-kpis { display: flex; gap: 10px; margin-bottom: 12px; }
.inv-kpi {
  flex: 1;
  background: var(--card);
  border: 1px solid var(--line);
  border-radius: 14px;
  padding: 14px 16px 12px 18px;
  position: relative;
  min-width: 0;
}
.inv-kpi::before {
  content: "";
  position: absolute;
  left: 8px;
  top: 14px;
  bottom: 14px;
  width: 3px;
  border-radius: 99px;
  background: var(--line);
}
.inv-kpi.accent::before { background: var(--accent); }
.inv-kpi.ok::before { background: #059669; }
.inv-kpi.bad::before { background: #e11d48; }
.inv-kpi.warn::before { background: #d97706; }
.inv-kpi .n { font-size: 28px; font-weight: 600; letter-spacing: -0.05em; line-height: 1; font-variant-numeric: tabular-nums; }
.inv-kpi .l { margin-top: 8px; font-size: 12px; font-weight: 600; color: var(--muted); }
.inv-card, .inv-plot {
  background: var(--card);
  border: 1px solid var(--line);
  border-radius: 16px;
  box-sizing: border-box;
}
.inv-card { padding: 12px 14px 14px; }
.inv-label {
  font-size: 11px;
  font-weight: 700;
  letter-spacing: 0.06em;
  text-transform: uppercase;
  color: var(--muted);
  margin: 0 0 8px;
}
.inv-shell .widget-dropdown > select,
.inv-shell input[type="number"],
.inv-shell .widget-select > select {
  border-radius: 10px !important;
  border: 1px solid #e5e7eb !important;
  background: #f8fafc !important;
  font-size: 13px !important;
  color: var(--ink) !important;
  min-height: 34px;
}
.inv-shell button.widget-toggle-button {
  border-radius: 10px !important;
  border: 1px solid #e5e7eb !important;
  background: #f8fafc !important;
  box-shadow: none !important;
  font-size: 13px !important;
  color: var(--ink) !important;
  min-height: 34px;
}
.inv-shell button.widget-toggle-button.mod-success {
  background: #059669 !important;
  border-color: #059669 !important;
  color: #fff !important;
}
.inv-shell select option:checked { background: #eef2ff; color: #312e81; }
.inv-meta {
  font-size: 11px;
  font-weight: 700;
  letter-spacing: 0.06em;
  text-transform: uppercase;
  color: var(--muted);
  margin: 0 2px 8px;
}
.inv-empty { padding: 28px 8px; text-align: center; color: var(--muted); font-size: 13px; }
.inv-legend { display: flex; flex-wrap: wrap; gap: 8px; align-items: center; margin-top: 8px; margin-bottom: 14px; }
.inv-agrupar { margin-bottom: 8px; }
.inv-legend-title {
  font-size: 11px;
  font-weight: 700;
  letter-spacing: 0.06em;
  text-transform: uppercase;
  color: var(--muted);
  margin-right: 4px;
}
.inv-key {
  display: inline-flex;
  align-items: center;
  gap: 6px;
  background: #f8fafc;
  border: 1px solid var(--line);
  border-radius: 999px;
  padding: 4px 10px 4px 8px;
  font-size: 12px;
  color: #374151;
}
.inv-key i { width: 8px; height: 8px; border-radius: 99px; display: inline-block; }
.inv-key b { font-weight: 600; color: var(--muted); font-variant-numeric: tabular-nums; }
.inv-shell .widget-html,
.inv-shell .widget-html-content {
  overflow: visible !important;
  max-width: 100%;
}
.inv-board {
  display: grid;
  grid-template-columns: repeat(auto-fit, minmax(min(100%, 300px), 1fr));
  gap: 12px;
  width: 100%;
  min-width: 0;
}
.inv-ns {
  border: 1px solid var(--line);
  border-radius: 14px;
  padding: 12px 14px 16px;
  background: #fff;
  min-width: 0;
  overflow: visible;
}
.inv-ns header { display: flex; justify-content: space-between; align-items: baseline; gap: 12px; margin-bottom: 10px; }
.inv-ns h3 { margin: 0; font-size: 14px; font-weight: 600; letter-spacing: -0.02em; }
.inv-ns header span { font-size: 12px; color: var(--muted); font-variant-numeric: tabular-nums; }
.inv-mount { display: grid; grid-template-columns: 92px 1fr 28px; gap: 8px; align-items: center; margin-bottom: 8px; }
.inv-mount-name { font-size: 12px; color: #374151; overflow: hidden; text-overflow: ellipsis; white-space: nowrap; }
.inv-mount:hover { position: relative; z-index: 3; }
.inv-stack { display: flex; height: 12px; border-radius: 99px; background: #f3f4f6; overflow: visible; }
.inv-seg { position: relative; display: block; height: 100%; min-width: 4px; }
.inv-seg:first-child { border-radius: 99px 0 0 99px; }
.inv-seg:last-child { border-radius: 0 99px 99px 0; }
.inv-seg:only-child { border-radius: 99px; }
.inv-seg::before {
  content: "";
  position: absolute;
  left: 0; right: 0;
  top: -8px; bottom: -8px;
}
.inv-seg::after {
  content: attr(data-tip);
  position: absolute;
  left: 50%;
  top: calc(100% + 8px);
  transform: translateX(-50%);
  padding: 4px 8px;
  border-radius: 8px;
  background: #111827;
  color: #fff;
  font-size: 12px;
  font-weight: 600;
  line-height: 1.3;
  white-space: nowrap;
  pointer-events: none;
  opacity: 0;
  z-index: 4;
}
.inv-seg:hover::after { opacity: 1; }
.inv-mount-n { font-size: 12px; text-align: right; color: var(--muted); font-variant-numeric: tabular-nums; }
.inv-table-wrap { overflow: auto; max-height: 480px; }
.inv-table.inv-grid { table-layout: fixed; width: max-content; min-width: 100%; }
.inv-table.inv-grid th { position: relative; vertical-align: bottom; }
.inv-sort {
  background: none; border: 0; padding: 0 26px 6px 0; margin: 0;
  font: inherit; font-size: 11px; font-weight: 700; letter-spacing: 0.04em;
  text-transform: uppercase; color: var(--muted); cursor: pointer; text-align: left;
}
.inv-col-x {
  position: absolute;
  top: 4px;
  right: 12px;
  z-index: 2;
  width: 18px;
  height: 18px;
  border: 0;
  border-radius: 999px;
  background: #e5e7eb;
  color: #374151;
  font-size: 14px;
  line-height: 16px;
  padding: 0;
  cursor: pointer;
  opacity: 0;
}
.inv-table.inv-grid th:hover .inv-col-x,
.inv-col-x:focus-visible { opacity: 1; }
.inv-col-x:hover { background: #ffe4e6; color: #be123c; }
.inv-filter {
  width: calc(100% - 8px); box-sizing: border-box; display: block;
  border: 1px solid #e5e7eb; border-radius: 8px; background: #fff;
  padding: 4px 6px; font-size: 12px; color: var(--ink);
}
.inv-resize {
  position: absolute; top: 0; right: 0; width: 8px; height: 100%;
  cursor: col-resize;
}
.inv-menu {
  position: fixed;
  z-index: 30;
  background: #fff;
  border: 1px solid #e5e7eb;
  border-radius: 10px;
  padding: 4px;
  max-height: 220px;
  overflow: auto;
}
.inv-menu button {
  display: flex;
  width: 100%;
  justify-content: space-between;
  gap: 12px;
  border: 0;
  background: transparent;
  border-radius: 8px;
  padding: 6px 8px;
  font-size: 12px;
  color: var(--ink);
  cursor: pointer;
  text-align: left;
}
.inv-menu button b { color: var(--muted); font-weight: 600; }
.inv-menu button:hover { background: #eef2ff; }
.inv-pager { display: flex; align-items: center; gap: 8px; margin-top: 10px; color: var(--muted); font-size: 12px; }
.inv-pager button {
  border: 1px solid #e5e7eb; background: #fff; border-radius: 10px;
  padding: 4px 10px; font-size: 12px; color: var(--ink); cursor: pointer;
}
.inv-pager button:disabled { opacity: 0.4; cursor: default; }
.inv-restore {
  display: flex;
  flex-wrap: wrap;
  gap: 8px;
  margin-top: 10px;
}
.inv-restore[hidden] { display: none; }
.inv-restore button {
  border: 1px solid #e5e7eb;
  background: #fff;
  border-radius: 999px;
  padding: 5px 12px;
  font-size: 12px;
  color: var(--ink);
  cursor: pointer;
}
.inv-restore button:hover {
  background: #eef2ff;
  border-color: #c7d2fe;
}
.inv-table { width: 100%; border-collapse: collapse; font-size: 12.5px; }
.inv-table th {
  position: sticky; top: 0; background: #f8fafc; text-align: left;
  font-size: 11px; letter-spacing: 0.04em; text-transform: uppercase; color: var(--muted);
  padding: 8px 10px; border-bottom: 1px solid var(--line); white-space: nowrap;
}
.inv-table td { padding: 8px 10px; border-bottom: 1px solid #f1f2f4; white-space: nowrap; color: #1f2937; }
.inv-table tbody tr:hover td { background: #f8fafc; }
.inv-pill {
  display: inline-block; padding: 1px 8px; border-radius: 999px;
  background: #f3f4f6; color: #4b5563; font-size: 11px; font-weight: 650;
}
.inv-pill-ok { background: #d1fae5; color: #047857; }
.inv-pill-bad { background: #ffe4e6; color: #be123c; }
.inv-pill-warn { background: #fef3c7; color: #b45309; }
.inv-muted { color: #9ca3af; }
</style>
""")
css_w.layout = widgets.Layout(height="0px", overflow="hidden", margin="0")
kpis = widgets.HTML()
tablero = widgets.HTML()
tabla = TablaInventario(page_size=PAGE_SIZE)
head = widgets.HTML('<div class="inv-head"><h2>Inventario KV</h2></div>')
group_w.layout = widgets.Layout(width="260px", margin="0 0 4px 0")
meta_btn.layout = widgets.Layout(width="260px", margin="0 0 14px 0")
campo_w.layout = widgets.Layout(width="260px", margin="0 0 18px 0")
agrupar = bloque("Agrupar", group_w)
agrupar.add_class("inv-agrupar")
agrupar.layout = widgets.Layout(width="260px", margin="0 0 20px 0", overflow="visible")
board_box = widgets.VBox(
    [agrupar, tablero],
    layout=widgets.Layout(width="100%", overflow="visible", margin="12px 0 0 0"),
)
board_box.add_class("inv-card")
campo = bloque("Campo", widgets.VBox([meta_btn, campo_w]))
campo.layout = widgets.Layout(width="260px", margin="0 0 8px 0", overflow="visible")
tabla_box = widgets.VBox(
    [campo, tabla],
    layout=widgets.Layout(width="100%", margin="12px 0 0 0"),
)
tabla_box.add_class("inv-card")
shell = widgets.VBox(
    [css_w, head, kpis, tabla_box, board_box],
    layout=widgets.Layout(width="100%"),
)
shell.add_class("inv-shell")
for widget in (group_w, campo_w, meta_btn):
    widget.observe(redraw, names="value")
tabla.observe(redraw, names=["column_filters", "column_exact"])
display(shell)
redraw()
